In [ ]:
import time
import numpy as np
import torch
from torch import nn
from d2l import torch as d2l


Helper functions: 
The first utility function allows us to register functions as methods in a class after the class has been created. In fact, we can do so even after we have created instances of the class! It allows us to split the implementation of a class into multiple code blocks.

In [ ]:
def add_to_class(Class):  #@save
    """Register functions as methods in created class."""
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

The second one is a utility class that saves all arguments in a class’s __init__ method as class attributes. This allows us to extend constructor call signatures implicitly without additional code.

In [ ]:
import inspect

class HyperParameters:  #@save
    """The base class of hyperparameters."""

    def save_hyperparameters(self, ignore=[]):
        frame = inspect.currentframe().f_back
        _, _, _, local_vars = inspect.getargvalues(frame)

        self.hparams = {
            k: v for k, v in local_vars.items()
            if k not in set(ignore + ['self'])
            and not k.startswith('_')
        }

        for k, v in self.hparams.items():
            setattr(self, k, v)

The final utility allows us to plot experiment progress interactively while it is going on.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from d2l import torch as d2l

class ProgressBoard(d2l.HyperParameters):  #@save
    """The board that plots data points in animation."""

    def __init__(self, xlabel=None, ylabel=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 ls=['-', '--', '-.', ':'], colors=['C0', 'C1', 'C2', 'C3'],
                 fig=None, axes=None, figsize=(3.5, 2.5), display=True):
        self.save_hyperparameters()

    def draw(self, x, y, label, every_n=1):
        if every_n < 1:
            raise ValueError("every_n must be at least 1")

        # Initialize storage on the first call
        if not hasattr(self, 'raw_points'):
            self.raw_points = {}
            self.data = {}

        if label not in self.raw_points:
            self.raw_points[label] = []
            self.data[label] = []

        # Collect a new point
        points = self.raw_points[label]
        points.append((x, y))

        # Wait until enough points have accumulated
        if len(points) < every_n:
            return

        # Average the accumulated points
        avg_x = sum(p[0] for p in points) / len(points)
        avg_y = sum(p[1] for p in points) / len(points)

        self.data[label].append((avg_x, avg_y))
        points.clear()

        if not self.display:
            return

        # Create the figure if necessary
        if self.fig is None or self.axes is None:
            self.fig, self.axes = plt.subplots(figsize=self.figsize)

        self.axes.clear()

        # Draw each labeled curve
        for i, (name, values) in enumerate(self.data.items()):
            xs = [p[0] for p in values]
            ys = [p[1] for p in values]

            self.axes.plot(
                xs, ys,
                label=name,
                linestyle=self.ls[i % len(self.ls)],
                color=self.colors[i % len(self.colors)]
            )

        # Configure the plot
        self.axes.set_xlabel(self.xlabel or 'x')
        self.axes.set_ylabel(self.ylabel or 'y')
        self.axes.set_xscale(self.xscale)
        self.axes.set_yscale(self.yscale)

        if self.xlim is not None:
            self.axes.set_xlim(self.xlim)
        if self.ylim is not None:
            self.axes.set_ylim(self.ylim)

        self.axes.legend()

        # Update the notebook output
        clear_output(wait=True)
        display(self.fig)

test that func:

In [ ]:
board = d2l.ProgressBoard('x')
for x in np.arange(0, 10, 0.1):
    board.draw(x, np.sin(x), 'sin', every_n=2)
    board.draw(x, np.cos(x), 'cos', every_n=10)

The Module class is the base class of all models we will implement. At the very least we need three methods. The first, __init__, stores the learnable parameters, the training_step method accepts a data batch to return the loss value, and finally, configure_optimizers returns the optimization method, or a list of them, that is used to update the learnable parameters. Optionally we can define validation_step to report the evaluation measures. Sometimes we put the code for computing the output into a separate forward method to make it more reusable.

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

class Module(nn.Module, d2l.HyperParameters):  #@save
    """The base class of models."""

    def __init__(self, plot_train_per_epoch=2, plot_valid_per_epoch=1):
        super().__init__()
        self.save_hyperparameters()
        self.board = d2l.ProgressBoard()

    def loss(self, y_hat, y):
        """Calculate mean squared error."""
        return nn.functional.mse_loss(y_hat, y.reshape(y_hat.shape))

    def forward(self, X):
        assert hasattr(self, 'net'), 'Neural network is defined'
        return self.net(X)

    def plot(self, key, value, train):
        """Plot a point in animation."""
        assert hasattr(self, 'trainer'), 'Trainer is not inited'
        self.board.xlabel = 'epoch'

        if train:
            x = self.trainer.train_batch_idx / \
                self.trainer.num_train_batches
            n = self.trainer.num_train_batches / \
                self.plot_train_per_epoch
        else:
            x = self.trainer.epoch + 1
            n = self.trainer.num_val_batches / \
                self.plot_valid_per_epoch

        self.board.draw(
            x,
            value.detach().cpu().numpy(),
            ('train_' if train else 'val_') + key,
            every_n=max(1, int(n))
        )

    def training_step(self, batch):
        l = self.loss(self(*batch[:-1]), batch[-1])
        self.plot('loss', l, train=True)
        return l

    def validation_step(self, batch):
        l = self.loss(self(*batch[:-1]), batch[-1])
        self.plot('loss', l, train=False)

    def configure_optimizers(self):
        """Configure stochastic gradient descent."""
        lr = getattr(self, 'lr', 0.01)
        return torch.optim.SGD(self.parameters(), lr=lr)

In [ ]:
class DataModule(d2l.HyperParameters):  #@save
    """The base class of data."""
    def __init__(self, root='../data', num_workers=4):
        self.save_hyperparameters()

    def get_dataloader(self, train):
        raise NotImplementedError

    def train_dataloader(self):
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        return self.get_dataloader(train=False)

In [ ]:
class Trainer(d2l.HyperParameters):  #@save
    """The base class for training models with data."""
    def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
        self.save_hyperparameters()
        assert num_gpus == 0, 'No GPU support yet'

    def prepare_data(self, data):
        self.train_dataloader = data.train_dataloader()
        self.val_dataloader = data.val_dataloader()
        self.num_train_batches = len(self.train_dataloader)
        self.num_val_batches = (len(self.val_dataloader)
                                if self.val_dataloader is not None else 0)

    def prepare_model(self, model):
        model.trainer = self
        model.board.xlim = [0, self.max_epochs]
        self.model = model

    def fit(self, model, data):
        self.prepare_data(data)
        self.prepare_model(model)
        self.optim = model.configure_optimizers()
        self.epoch = 0
        self.train_batch_idx = 0
        self.val_batch_idx = 0
        for self.epoch in range(self.max_epochs):
            self.fit_epoch()

    def fit_epoch(self):
        raise NotImplementedError